In [35]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

data_path = Path("../data/Telco-Customer-Churn.csv")
df = pd.read_csv(data_path)
print(df.shape)
print(df.dtypes.value_counts())

(7043, 21)
str        18
int64       2
float64     1
Name: count, dtype: int64


In [36]:
# ---- Data-specific cleanup (types only, no statistics) ----
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.drop(columns=["customerID"])  # identifiers are never features

# ---- Decision sentence ----
# Because of this model, the retention team will call any customer whose
# predicted churn probability exceeds the chosen threshold, within 3 days
# of their next billing date.

# ---- Target definition ----
TARGET = "Churn"
X, y = df.drop(columns=[TARGET]), (df[TARGET] == "Yes").astype(int)

print("Features:", X.shape[1])
print("Base rate (churn %):", y.mean().round(4))

Features: 19
Base rate (churn %): 0.2654


In [37]:
# ---- Sealed 60/20/20 stratified split ----
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=42
)
X_val, X_te, y_val, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=42
)

print(f"Train: {len(X_tr)} ({y_tr.mean():.2%} churn)")
print(f"Validation: {len(X_val)} ({y_val.mean():.2%} churn)")
print(f"Test: {len(X_te)} ({y_te.mean():.2%} churn)")
print("\n⚠️ TEST SET SEALED — do not touch X_te / y_te again until Stage 6")

Train: 4225 (26.53% churn)
Validation: 1409 (26.54% churn)
Test: 1409 (26.54% churn)

⚠️ TEST SET SEALED — do not touch X_te / y_te again until Stage 6


In [38]:
# ---- Baseline 1: Majority class ----
majority_pred = pd.Series(0, index=y_val.index)
majority_acc = (majority_pred == y_val).mean()
print("Majority class baseline accuracy:", round(majority_acc, 4))

Majority class baseline accuracy: 0.7346


In [39]:
# ---- Baseline 2: Simple business rule ----
# Rule: flag anyone on a month-to-month contract with no tech support
rule_pred = ((X_val["Contract"] == "Month-to-month") & (X_val["TechSupport"] == "No")).astype(int)
rule_acc = (rule_pred == y_val).mean()
print("Business rule baseline accuracy:", round(rule_acc, 4))

Business rule baseline accuracy: 0.7424


In [40]:
# ---- Baseline 3: Logistic regression on three obvious features ----
from sklearn.linear_model import LogisticRegression

X_tr_simple = pd.DataFrame({
    "tenure": X_tr["tenure"],
    "MonthlyCharges": X_tr["MonthlyCharges"],
    "is_month_to_month": (X_tr["Contract"] == "Month-to-month").astype(int)
})
X_val_simple = pd.DataFrame({
    "tenure": X_val["tenure"],
    "MonthlyCharges": X_val["MonthlyCharges"],
    "is_month_to_month": (X_val["Contract"] == "Month-to-month").astype(int)
})

logreg = LogisticRegression(random_state=42)
logreg.fit(X_tr_simple, y_tr)
logreg_pred = logreg.predict(X_val_simple)
logreg_acc = (logreg_pred == y_val).mean()
print("Logistic regression (3 features) baseline accuracy:", round(logreg_acc, 4))

Logistic regression (3 features) baseline accuracy: 0.7906


In [41]:
# ---- Reusable preprocessing pipeline ----
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

print("Pipeline built successfully.")

Numeric features: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Pipeline built successfully.


C:\Users\samya\AppData\Local\Temp\ipykernel_14596\1140875323.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=["object"]).columns.tolist()


In [42]:
# ---- Stage 2: Regression detour on TotalCharges ----
TARGET_REG = "TotalCharges"
X_reg, y_reg = df.drop(columns=[TARGET_REG]), df[TARGET_REG]

# Same sealed split logic, but for the regression target
X_tr_r, X_tmp_r, y_tr_r, y_tmp_r = train_test_split(
    X_reg, y_reg, test_size=0.40, random_state=42
)
X_val_r, X_te_r, y_val_r, y_te_r = train_test_split(
    X_tmp_r, y_tmp_r, test_size=0.50, random_state=42
)

# Drop rows where TotalCharges is missing (the 11 known missing values)
train_mask = y_tr_r.notna()
val_mask = y_val_r.notna()
X_tr_r, y_tr_r = X_tr_r.loc[train_mask], y_tr_r.loc[train_mask]
X_val_r, y_val_r = X_val_r.loc[val_mask], y_val_r.loc[val_mask]

print(f"Train: {len(X_tr_r)}  Val: {len(X_val_r)}  Test: {len(X_te_r)}")
print(f"Target mean (train): {y_tr_r.mean():.2f}")

Train: 4217  Val: 1409  Test: 1409
Target mean (train): 2297.64


In [43]:
# ---- Mean baseline (regression) ----
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

base = DummyRegressor(strategy="mean").fit(X_tr_r, y_tr_r)
pred_base = base.predict(X_val_r)

mae_base = mean_absolute_error(y_val_r, pred_base)
rmse_base = mean_squared_error(y_val_r, pred_base) ** 0.5
r2_base = r2_score(y_val_r, pred_base)

print(f"Mean baseline — MAE: {mae_base:.2f}  RMSE: {rmse_base:.2f}  R2: {r2_base:.3f}")

Mean baseline — MAE: 1918.02  RMSE: 2275.67  R2: -0.000


In [44]:
# ---- Four regressors, identical folds ----
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_validate, KFold

cv = KFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["neg_mean_absolute_error", "neg_root_mean_squared_error", "r2"]

candidates = {
    "mean baseline": DummyRegressor(strategy="mean"),
    "linear": LinearRegression(),
    "ridge": Ridge(alpha=1.0),
    "lasso": Lasso(alpha=0.1, max_iter=5000),
    "random forest": RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=1),
}

num_r = X_tr_r.select_dtypes(include="number").columns.tolist()
cat_r = X_tr_r.select_dtypes(exclude="number").columns.tolist()

pre_r = ColumnTransformer(transformers=[
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_r),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_r),
])

cv_results = {
    name: cross_validate(
        Pipeline([("pre", pre_r), ("reg", model)]),
        X_tr_r, y_tr_r,
        cv=cv, scoring=scoring,
        return_train_score=True, n_jobs=1, error_score="raise",
    )
    for name, model in candidates.items()
}

def summarize_cv(results_dict):
    rows = []
    for name, r in results_dict.items():
        rows.append({
            "model": name,
            "MAE": -r["test_neg_mean_absolute_error"].mean(),
            "RMSE": -r["test_neg_root_mean_squared_error"].mean(),
            "R2": r["test_r2"].mean(),
            "train_MAE": -r["train_neg_mean_absolute_error"].mean(),
        })
    summary = pd.DataFrame(rows).sort_values("RMSE")
    print(summary.to_string(index=False, float_format=lambda v: f"{v:,.3f}"))
    return summary

summary_with_leak = summarize_cv(cv_results)

        model       MAE      RMSE     R2  train_MAE
random forest    55.954    84.414  0.999     20.860
        ridge   554.417   689.518  0.908    550.910
        lasso   554.362   689.526  0.908    550.905
       linear   554.459   689.666  0.908    550.884
mean baseline 1,905.056 2,270.398 -0.001  1,904.709


In [45]:
# ---- Leakage investigation: drop MonthlyCharges and re-run ----
print("=== Re-running WITHOUT MonthlyCharges (leakage check) ===\n")

X_tr_r_noleak = X_tr_r.drop(columns=["MonthlyCharges"])

num_r_noleak = X_tr_r_noleak.select_dtypes(include="number").columns.tolist()
cat_r_noleak = X_tr_r_noleak.select_dtypes(exclude="number").columns.tolist()

pre_r_noleak = ColumnTransformer(transformers=[
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_r_noleak),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_r_noleak),
])

cv_results_noleak = {
    name: cross_validate(
        Pipeline([("pre", pre_r_noleak), ("reg", model)]),
        X_tr_r_noleak, y_tr_r,
        cv=cv, scoring=scoring,
        return_train_score=True, n_jobs=1, error_score="raise",
    )
    for name, model in candidates.items()
}

summary_no_leak = summarize_cv(cv_results_noleak)

=== Re-running WITHOUT MonthlyCharges (leakage check) ===

        model       MAE      RMSE     R2  train_MAE
random forest   101.491   162.980  0.995     37.841
        lasso   554.722   689.705  0.908    551.216
        ridge   554.746   689.709  0.908    551.235
       linear   554.734   689.714  0.908    551.219
mean baseline 1,905.056 2,270.398 -0.001  1,904.709


## Leakage Investigation: TotalCharges ≈ tenure × MonthlyCharges

Random Forest achieved a suspiciously high R² of 0.999 in the initial four-model comparison. This is a sign of data leakage: `TotalCharges` is nearly the product of `tenure` and `MonthlyCharges`, so a flexible model can reconstruct the target almost exactly rather than learning genuine customer behavior patterns.

**Investigation:** Dropped `MonthlyCharges` and re-ran the identical comparison.

**Finding:** Random Forest's R² only dropped from 0.999 to 0.995 (MAE rose from 55.91 to 101.55), even without `MonthlyCharges` directly available. This is because service-related categorical features (InternetService, StreamingTV, OnlineSecurity, TechSupport, etc.) collectively encode pricing information — these add-ons are what determine a customer's monthly bill. Random Forest is nonlinear enough to reconstruct an approximation of `MonthlyCharges` from these columns and effectively re-derive the tenure × price relationship internally.

The linear models (Linear, Ridge, Lasso) were essentially unaffected by dropping `MonthlyCharges` (R² stayed at 0.908 in both cases), since they cannot easily model a multiplicative relationship between two features regardless of which columns are available.

**Conclusion:** The leakage here is structural — it stems from how `TotalCharges` is defined, not from a single removable column. Both regression tables are reported above for transparency.

In [46]:
# ---- Residual plot and diagnosis ----
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

Path("../reports").mkdir(exist_ok=True)

pipe_lasso = Pipeline([("pre", pre_r), ("reg", candidates["lasso"])])
pred_lasso = pipe_lasso.fit(X_tr_r, y_tr_r).predict(X_val_r)

resid = y_val_r - pred_lasso
abs_resid = resid.abs()

mae_lasso = abs_resid.mean()
rmse_lasso = (resid ** 2).mean() ** 0.5

print(f"MAE : {mae_lasso:.2f}")
print(f"RMSE: {rmse_lasso:.2f}")
print(f"Mean residual: {resid.mean():.2f}")
print(f"|residual| vs prediction correlation: {pd.Series(pred_lasso, index=y_val_r.index).corr(abs_resid):.3f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(pred_lasso, resid, s=8, alpha=0.4)
ax[0].axhline(0, color="crimson", lw=1)
ax[0].set_xlabel("predicted"); ax[0].set_ylabel("residual")
ax[0].set_title("Residuals vs predicted")
ax[1].hist(resid, bins=40)
ax[1].set_title("Residual distribution")
fig.tight_layout()
fig.savefig("../reports/residuals.png", dpi=120)
print("\nSaved to reports/residuals.png")

MAE : 568.73
RMSE: 700.83
Mean residual: -14.41
|residual| vs prediction correlation: 0.037

Saved to reports/residuals.png


## Stage 2 Write-up Notes

**Primary metric:** MAE is the primary metric for this problem, since it reflects the typical dollar-value prediction error without letting a small number of high-tenure, high-bill customers dominate the score the way RMSE would.

**RMSE vs MAE ratio:** RMSE is about 1.23× MAE (700.83 / 568.73), which is below the 1.5 threshold. This means prediction errors are fairly evenly distributed across customers, rather than being driven by a small number of extreme outlier misses.

**Residual pattern:** Mean residual is close to zero (-14.41) and error magnitude does not grow with prediction level (correlation 0.037), indicating a random scatter around zero with no strong systematic bias — no clear sign of heteroscedasticity or a missed nonlinear relationship.

In [47]:
# ---- Error analysis by segment ----
print("=== Error analysis by segment ===")
seg = X_val_r.copy()
seg["abs_err"] = np.abs(y_val_r - pred_lasso)
overall = seg["abs_err"].mean()

frames = []
for col in X_val_r.select_dtypes(exclude="number").columns:
    g = (seg.groupby(col)["abs_err"].agg(n="size", mae="mean")
         .assign(feature=col, gap=lambda d: d["mae"] - overall)
         .reset_index().rename(columns={col: "segment"}))
    frames.append(g)

for col in X_val_r.select_dtypes(include="number").columns[:5]:
    q = pd.qcut(seg[col], 4, duplicates="drop")
    g = (seg.groupby(q, observed=True)["abs_err"]
         .agg(n="size", mae="mean").assign(feature=col, gap=lambda d: d["mae"] - overall)
         .reset_index().rename(columns={col: "segment"}))
    g["segment"] = g["segment"].astype(str)
    frames.append(g)

table = pd.concat(frames)
table["impact"] = table["n"] * table["gap"]
print(f"overall MAE {overall:.2f}")
print(table.sort_values("impact", ascending=False).head(10).to_string(index=False))
table.to_csv("../reports/error_by_segment.csv", index=False)
print("\nSaved to reports/error_by_segment.csv")

=== Error analysis by segment ===
overall MAE 568.73
                    segment   n        mae          feature        gap       impact
(18.698999999999998, 35.45] 353 838.985569   MonthlyCharges 270.260271 95401.875777
               (56.0, 72.0] 343 799.733311           tenure 231.008013 79235.748612
        No internet service 303 820.450609     OnlineBackup 251.725311 76272.769348
                         No 303 820.450609  InternetService 251.725311 76272.769348
        No internet service 303 820.450609      TechSupport 251.725311 76272.769348
        No internet service 303 820.450609 DeviceProtection 251.725311 76272.769348
        No internet service 303 820.450609  StreamingMovies 251.725311 76272.769348
        No internet service 303 820.450609      StreamingTV 251.725311 76272.769348
        No internet service 303 820.450609   OnlineSecurity 251.725311 76272.769348
            (89.95, 118.75] 352 730.666009   MonthlyCharges 161.940711 57003.130309

Saved to reports/error

In [48]:
# ---- Stage 3: Classification — Dummy + 4 classifiers, identical folds ----
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              confusion_matrix, roc_auc_score, average_precision_score)

cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring_clf = ["average_precision", "roc_auc"]

clf_candidates = {
    "dummy": DummyClassifier(strategy="most_frequent"),
    "logistic": LogisticRegression(max_iter=1000, random_state=42),
    "knn": KNeighborsClassifier(),
    "gaussianNB": GaussianNB(),
    "svm": SVC(probability=True, random_state=42),
}

# Reuse the same preprocessor structure as Stage 1, applied to the classification split
num_c = X_tr.select_dtypes(include="number").columns.tolist()
cat_c = X_tr.select_dtypes(exclude="number").columns.tolist()

pre_c = ColumnTransformer(transformers=[
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_c),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_c),
])

cv_clf_results = {}
for name, model in clf_candidates.items():
    r = cross_validate(
        Pipeline([("pre", pre_c), ("clf", model)]),
        X_tr, y_tr,
        cv=cv_clf, scoring=scoring_clf,
        return_train_score=True, n_jobs=1, error_score="raise",
    )
    cv_clf_results[name] = r
    print(f"{name:10s}  PR-AUC: {r['test_average_precision'].mean():.3f}  "
          f"ROC-AUC: {r['test_roc_auc'].mean():.3f}  "
          f"train PR-AUC: {r['train_average_precision'].mean():.3f}")

dummy       PR-AUC: 0.265  ROC-AUC: 0.500  train PR-AUC: 0.265
logistic    PR-AUC: 0.663  ROC-AUC: 0.845  train PR-AUC: 0.669
knn         PR-AUC: 0.525  ROC-AUC: 0.787  train PR-AUC: 0.716
gaussianNB  PR-AUC: 0.619  ROC-AUC: 0.821  train PR-AUC: 0.618


c:\Users\samya\miniconda3\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\samya\miniconda3\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\samya\miniconda3\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\samya\miniconda3\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. 

svm         PR-AUC: 0.644  ROC-AUC: 0.804  train PR-AUC: 0.724


In [49]:
# ---- Best model: confusion matrix and full metrics on validation ----
best_pipe = Pipeline([("pre", pre_c), ("clf", clf_candidates["logistic"])])
best_pipe.fit(X_tr, y_tr)

proba_val = best_pipe.predict_proba(X_val)[:, 1]
pred_val_05 = (proba_val >= 0.5).astype(int)

cm = confusion_matrix(y_val, pred_val_05)
print("Confusion matrix (threshold=0.5):")
print(cm)
print(f"\nBase rate: {y_val.mean():.3f}")
print(f"Precision: {precision_score(y_val, pred_val_05):.3f}")
print(f"Recall:    {recall_score(y_val, pred_val_05):.3f}")
print(f"F1:        {f1_score(y_val, pred_val_05):.3f}")
print(f"ROC-AUC:   {roc_auc_score(y_val, proba_val):.3f}")
print(f"PR-AUC:    {average_precision_score(y_val, proba_val):.3f}")

Confusion matrix (threshold=0.5):
[[939  96]
 [152 222]]

Base rate: 0.265
Precision: 0.698
Recall:    0.594
F1:        0.642
ROC-AUC:   0.860
PR-AUC:    0.678


In [50]:
# ---- Threshold sweep ----
thresholds = np.linspace(0.05, 0.95, 19)
precisions, recalls, f1s = [], [], []

for t in thresholds:
    pred_t = (proba_val >= t).astype(int)
    precisions.append(precision_score(y_val, pred_t, zero_division=0))
    recalls.append(recall_score(y_val, pred_t, zero_division=0))
    f1s.append(f1_score(y_val, pred_t, zero_division=0))

best_f1_idx = np.argmax(f1s)
best_f1_threshold = thresholds[best_f1_idx]

print(f"F1 peaks at threshold {best_f1_threshold:.2f} (F1 = {f1s[best_f1_idx]:.3f})")
print(f"F1 at threshold 0.50: {f1s[np.argmin(np.abs(thresholds - 0.5))]:.3f}")
print(f"Distance from 0.5: {abs(best_f1_threshold - 0.5):.2f}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(thresholds, precisions, label="Precision", marker="o", markersize=3)
ax.plot(thresholds, recalls, label="Recall", marker="o", markersize=3)
ax.plot(thresholds, f1s, label="F1", marker="o", markersize=3)
ax.axvline(best_f1_threshold, color="red", linestyle="--", alpha=0.5, label=f"F1 peak ({best_f1_threshold:.2f})")
ax.set_xlabel("Threshold"); ax.set_ylabel("Score")
ax.set_title("Precision, Recall, F1 vs Threshold")
ax.legend()
fig.tight_layout()
fig.savefig("../reports/threshold_sweep.png", dpi=120)
print("\nSaved to reports/threshold_sweep.png")

F1 peaks at threshold 0.40 (F1 = 0.645)
F1 at threshold 0.50: 0.642
Distance from 0.5: 0.10

Saved to reports/threshold_sweep.png


In [51]:
# ---- Cost-optimal threshold ----
def total_cost(y_true, proba, threshold, fn_cost, fp_cost):
    pred = (proba >= threshold).astype(int)
    fn = ((y_true == 1) & (pred == 0)).sum()
    fp = ((y_true == 0) & (pred == 1)).sum()
    return fn * fn_cost + fp * fp_cost

# Cost ratio: missing a churner costs 10x more than a wasted retention offer
# (retention offer ~ $25, lost customer lifetime value ~ $250+)
FN_COST = 10
FP_COST = 1

thresholds_fine = np.linspace(0.05, 0.95, 91)
costs = [total_cost(y_val, proba_val, t, FN_COST, FP_COST) for t in thresholds_fine]
best_idx = np.argmin(costs)
best_threshold = thresholds_fine[best_idx]

print(f"Cost ratio FN:FP = {FN_COST}:{FP_COST}")
print(f"Cost-optimal threshold: {best_threshold:.2f}")
print(f"Cost at optimal threshold: {costs[best_idx]:.0f}")
print(f"Cost at default (0.5): {total_cost(y_val, proba_val, 0.5, FN_COST, FP_COST):.0f}")

print("\n=== Sensitivity check across cost ratios ===")
for ratio in [5, 10, 20]:
    costs_r = [total_cost(y_val, proba_val, t, ratio, 1) for t in thresholds_fine]
    best_t_r = thresholds_fine[np.argmin(costs_r)]
    print(f"FN:FP = {ratio}:1  ->  optimal threshold: {best_t_r:.2f}")

Cost ratio FN:FP = 10:1
Cost-optimal threshold: 0.10
Cost at optimal threshold: 682
Cost at default (0.5): 1616

=== Sensitivity check across cost ratios ===
FN:FP = 5:1  ->  optimal threshold: 0.17
FN:FP = 10:1  ->  optimal threshold: 0.10
FN:FP = 20:1  ->  optimal threshold: 0.06


## Cost-Optimal Threshold

Assumed cost ratio: missing a churner (FN) costs 10× more than a wasted retention offer (FP), reflecting the gap between a lost customer's lifetime value (~$250+) and the cost of a retention offer (~$25).

At this ratio, the cost-minimizing threshold is **0.10** — far below the default 0.5 — and reduces total business cost from 1,616 (at threshold 0.5) to 682, a 58% reduction.

**Sensitivity check:** As the assumed cost ratio increases (meaning missed churners are treated as more costly relative to false alarms), the optimal threshold consistently drops:
- FN:FP = 5:1  → threshold 0.17
- FN:FP = 10:1 → threshold 0.10
- FN:FP = 20:1 → threshold 0.06

This confirms the threshold recommendation is directionally stable across a range of reasonable cost assumptions — even if the exact business costs aren't known precisely, a low threshold (well under 0.5) is consistently favored whenever missing a churner is treated as meaningfully more costly than a false alarm.

In [52]:
# ---- Imbalance strategies: nothing, threshold tuning, class_weight, SMOTE ----
from imblearn.pipeline import Pipeline as ImPipeline
from imblearn.over_sampling import SMOTE

def report_strategy(name, y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    p = precision_score(y_true, pred, zero_division=0)
    r = recall_score(y_true, pred, zero_division=0)
    f1 = f1_score(y_true, pred, zero_division=0)
    print(f"{name:20s} t={threshold:.2f}  P={p:.3f}  R={r:.3f}  F1={f1:.3f}")

# 1. Nothing — plain logistic regression, default threshold
pipe_plain = Pipeline([("pre", pre_c), ("clf", LogisticRegression(max_iter=1000, random_state=42))])
pipe_plain.fit(X_tr, y_tr)
proba_plain = pipe_plain.predict_proba(X_val)[:, 1]
report_strategy("1. Nothing", y_val, proba_plain, threshold=0.5)

# 2. Threshold tuning — same model, cost-optimal threshold from before
report_strategy("2. Threshold tuned", y_val, proba_plain, threshold=best_threshold)

# 3. class_weight="balanced"
pipe_weighted = Pipeline([("pre", pre_c), ("clf", LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced"))])
pipe_weighted.fit(X_tr, y_tr)
proba_weighted = pipe_weighted.predict_proba(X_val)[:, 1]
report_strategy("3. Class weight", y_val, proba_weighted, threshold=0.5)

# 4. SMOTE — resampling happens INSIDE the pipeline, only on training folds
pipe_smote = ImPipeline([
    ("pre", pre_c),
    ("smote", SMOTE(random_state=42)),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])
pipe_smote.fit(X_tr, y_tr)
proba_smote = pipe_smote.predict_proba(X_val)[:, 1]
report_strategy("4. SMOTE", y_val, proba_smote, threshold=0.5)

1. Nothing           t=0.50  P=0.698  R=0.594  F1=0.642
2. Threshold tuned   t=0.10  P=0.399  R=0.963  F1=0.564
3. Class weight      t=0.50  P=0.517  R=0.821  F1=0.634
4. SMOTE             t=0.50  P=0.518  R=0.818  F1=0.634


## Imbalance Strategy Comparison

| Strategy | Threshold | Precision | Recall | F1 |
|---|---|---|---|---|
| Nothing | 0.50 | 0.698 | 0.594 | 0.642 |
| Threshold tuned | 0.10 | 0.399 | 0.963 | 0.564 |
| Class weight | 0.50 | 0.517 | 0.821 | 0.634 |
| SMOTE | 0.50 | 0.518 | 0.818 | 0.634 |

By F1 alone, doing nothing performs best. However, F1 treats precision and recall as equally important, which doesn't match this business's actual costs (missing a churner is assumed to cost 10x more than a wasted retention offer). Under that cost structure, the threshold-tuned model — despite its lower F1 — is the right choice: it catches 96.3% of churners, which is what the retention program is actually trying to do.

Class weighting and SMOTE land in nearly identical territory (P≈0.52, R≈0.82) — two different mechanisms converging on a similar trade-off. Neither clearly beats simple threshold tuning on the plain model, suggesting the extra complexity of resampling or reweighting isn't buying much here beyond what threshold tuning already achieves for free.

**Recommendation:** Use the plain logistic regression pipeline with a tuned threshold (0.10), rather than adding SMOTE or class weighting — simpler, and performs comparably or better once the actual business cost ratio is accounted for.

In [53]:
# ---- Stage 4: Add Random Forest and Gradient Boosting to the comparison ----
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

clf_candidates_ensemble = {
    "dummy": DummyClassifier(strategy="most_frequent"),
    "logistic": LogisticRegression(max_iter=1000, random_state=42),
    "knn": KNeighborsClassifier(),
    "gaussianNB": GaussianNB(),
    "svm": SVC(probability=True, random_state=42),
    "random forest": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=1),
    "gradient boosting": HistGradientBoostingClassifier(random_state=42),
}

cv_ensemble_results = {}
for name, model in clf_candidates_ensemble.items():
    r = cross_validate(
        Pipeline([("pre", pre_c), ("clf", model)]),
        X_tr, y_tr,
        cv=cv_clf, scoring=scoring_clf,
        return_train_score=True, n_jobs=1, error_score="raise",
    )
    cv_ensemble_results[name] = r
    print(f"{name:20s}  PR-AUC: {r['test_average_precision'].mean():.3f}  "
          f"ROC-AUC: {r['test_roc_auc'].mean():.3f}  "
          f"train PR-AUC: {r['train_average_precision'].mean():.3f}")

dummy                 PR-AUC: 0.265  ROC-AUC: 0.500  train PR-AUC: 0.265
logistic              PR-AUC: 0.663  ROC-AUC: 0.845  train PR-AUC: 0.669
knn                   PR-AUC: 0.525  ROC-AUC: 0.787  train PR-AUC: 0.716
gaussianNB            PR-AUC: 0.619  ROC-AUC: 0.821  train PR-AUC: 0.618


c:\Users\samya\miniconda3\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\samya\miniconda3\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\samya\miniconda3\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\samya\miniconda3\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. 

svm                   PR-AUC: 0.644  ROC-AUC: 0.804  train PR-AUC: 0.724
random forest         PR-AUC: 0.624  ROC-AUC: 0.827  train PR-AUC: 1.000
gradient boosting     PR-AUC: 0.641  ROC-AUC: 0.833  train PR-AUC: 0.929


## Ensemble Comparison Findings

Adding Random Forest and Gradient Boosting to the comparison did not improve on Logistic Regression's validation PR-AUC (0.663). Random Forest achieved a perfect 1.000 PR-AUC on training data but only 0.624 on validation — a 0.376 gap, indicating severe overfitting despite using 300 trees. Gradient Boosting showed a smaller but still notable gap (0.929 train vs 0.641 validation).

This suggests the churn signal in this dataset is largely linear/additive (contract type, tenure, and a handful of service features dominate), so tree-based ensembles' capacity to model complex nonlinear interactions isn't being rewarded — it's mostly finding ways to overfit noise in the training set instead.

**Conclusion:** Logistic Regression remains the best-performing model by validation PR-AUC, and is also the most honest (smallest train/validation gap).

In [54]:
# ---- Boosting overfitting curve ----
from sklearn.metrics import log_loss

# Split training data further for early-stopping style monitoring
X_fit, X_es, y_fit, y_es = train_test_split(
    X_tr, y_tr, test_size=0.15, stratify=y_tr, random_state=42
)

X_fit_enc = pre_c.fit_transform(X_fit, y_fit)
X_es_enc = pre_c.transform(X_es)

gb_monitor = HistGradientBoostingClassifier(
    max_iter=600, random_state=42, early_stopping=False,
    validation_fraction=None
)
gb_monitor.fit(X_fit_enc, y_fit)

train_losses = []
held_out_losses = []
n_iterations = range(1, len(gb_monitor.train_score_) + 1)

# HistGradientBoostingClassifier tracks train_score_ automatically when early_stopping is on;
# to get per-iteration loss on held-out data, use staged predictions manually:
from sklearn.ensemble import HistGradientBoostingClassifier as HGB

gb_staged = HGB(max_iter=600, random_state=42, early_stopping=True,
                validation_fraction=0.15, n_iter_no_change=600, tol=0)
gb_staged.fit(X_fit_enc, y_fit)

train_loss_curve = 1 - np.array(gb_staged.train_score_)
val_loss_curve = 1 - np.array(gb_staged.validation_score_)

best_round = np.argmin(val_loss_curve) + 1
print(f"Best round (lowest held-out loss): {best_round}")
print(f"Held-out loss at best round: {val_loss_curve[best_round-1]:.4f}")
print(f"Held-out loss at final round ({len(val_loss_curve)}): {val_loss_curve[-1]:.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(train_loss_curve)+1), train_loss_curve, label="Train")
ax.plot(range(1, len(val_loss_curve)+1), val_loss_curve, label="Held-out")
ax.axvline(best_round, color="red", linestyle="--", alpha=0.5, label=f"Best round: {best_round}")
ax.set_xlabel("Boosting Rounds"); ax.set_ylabel("Loss (1 - AUC-based score)")
ax.set_title("Manual Early Stopping: Training vs Held-out Loss")
ax.legend()
fig.tight_layout()
fig.savefig("../reports/boosting_curve.png", dpi=120)
print("\nSaved to reports/boosting_curve.png")

Best round (lowest held-out loss): 44
Held-out loss at best round: 1.4352
Held-out loss at final round (601): 1.7243

Saved to reports/boosting_curve.png


## Boosting Overfitting Curve

Trained gradient boosting for 600 rounds while tracking held-out loss at each round. The optimal stopping point is round **44** — after this point, held-out loss increases (1.44 → 1.72 by round 600) even though training loss would continue to decrease, the classic overfitting signature for boosted trees.

This confirms early stopping is essential: an unconstrained boosting model trained for a fixed large number of rounds will overfit substantially past its optimal point.

In [55]:
import optuna

In [56]:
# ---- Optuna study: hyperparameter tuning with a declared budget ----
import optuna
from sklearn.model_selection import cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)  # quieter output

N_TRIALS = 60  # budget declared in advance

def objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 15, 63),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 100),
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-3, 10.0, log=True),
    }
    model = HistGradientBoostingClassifier(random_state=42, **params)
    pipe = Pipeline([("pre", pre_c), ("clf", model)])
    scores = cross_val_score(pipe, X_tr, y_tr, cv=cv_clf, scoring="average_precision", n_jobs=1)
    return scores.mean()

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=N_TRIALS)

print(f"Trial budget: {N_TRIALS}")
print(f"Search space: {list(study.best_params.keys())}")
print(f"Best CV PR-AUC: {study.best_value:.4f}")
print(f"Best config: {study.best_params}")
print(f"\nSpread of trial scores: min={min(t.value for t in study.trials):.4f}, "
      f"max={max(t.value for t in study.trials):.4f}, "
      f"std={np.std([t.value for t in study.trials]):.4f}")

Trial budget: 60
Search space: ['learning_rate', 'max_leaf_nodes', 'min_samples_leaf', 'l2_regularization']
Best CV PR-AUC: 0.6687
Best config: {'learning_rate': 0.021961044551154225, 'max_leaf_nodes': 18, 'min_samples_leaf': 80, 'l2_regularization': 0.7701825582004878}

Spread of trial scores: min=0.6170, max=0.6687, std=0.0094


In [57]:
# ---- Winner's curse experiment ----
trial_counts = [10, 25, 50, 60]
best_scores_at_n = []

sorted_trials = sorted(study.trials, key=lambda t: t.number)

for n in trial_counts:
    subset = [t.value for t in sorted_trials[:n]]
    best_scores_at_n.append(max(subset))

# Now measure the TRUE held-out performance of the actual best model
best_model = HistGradientBoostingClassifier(random_state=42, **study.best_params)
best_pipe_tuned = Pipeline([("pre", pre_c), ("clf", best_model)])
best_pipe_tuned.fit(X_tr, y_tr)
proba_val_tuned = best_pipe_tuned.predict_proba(X_val)[:, 1]
held_out_score = average_precision_score(y_val, proba_val_tuned)

print("Trials used | Best CV score | Held-out score | Gap")
for n, cv_score in zip(trial_counts, best_scores_at_n):
    gap = cv_score - held_out_score
    print(f"{n:11d} | {cv_score:.4f}         | {held_out_score:.4f}          | {gap:.4f}")

print(f"\nFinal tuned model's true validation PR-AUC: {held_out_score:.4f}")
print(f"CV reported: {study.best_value:.4f}")
print(f"Winner's curse gap: {study.best_value - held_out_score:.4f}")

Trials used | Best CV score | Held-out score | Gap
         10 | 0.6664         | 0.6723          | -0.0059
         25 | 0.6664         | 0.6723          | -0.0059
         50 | 0.6683         | 0.6723          | -0.0040
         60 | 0.6687         | 0.6723          | -0.0036

Final tuned model's true validation PR-AUC: 0.6723
CV reported: 0.6687
Winner's curse gap: -0.0036


## Winner's Curse Experiment

Measured the gap between the best cross-validation score at increasing trial counts (10, 25, 50, 60) and the true held-out validation score of the final tuned model.

**Result:** The gap came out slightly negative (-0.0036) — the held-out score (0.6723) was marginally higher than the best CV score (0.6687), rather than lower as the winner's-curse effect typically predicts.

**Honest interpretation:** This gap is smaller than the trial-to-trial standard deviation observed during tuning (0.0094), so it likely reflects ordinary fold-to-fold noise rather than evidence that tuning avoided the winner's curse entirely. With a 60-trial budget on a relatively flat search space (CV scores ranged only from 0.617 to 0.669), the optimism from selecting "the best" trial appears to be small relative to natural variance in this dataset. This does not mean winner's curse doesn't apply in general — only that its effect here is within noise, and should not be over-interpreted as a positive result.

In [58]:
# ---- Stage 5: Impurity vs Permutation Importance, with a random ID control ----
from sklearn.inspection import permutation_importance

# Plant a random high-cardinality control column
X_tr_control = X_tr.copy()
X_val_control = X_val.copy()
np.random.seed(42)
X_tr_control["random_id"] = np.random.randint(0, 100000, size=len(X_tr_control))
X_val_control["random_id"] = np.random.randint(0, 100000, size=len(X_val_control))

num_ctrl = X_tr_control.select_dtypes(include="number").columns.tolist()
cat_ctrl = X_tr_control.select_dtypes(exclude="number").columns.tolist()

pre_ctrl = ColumnTransformer(transformers=[
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_ctrl),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_ctrl),
])

rf_ctrl = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=1)
pipe_ctrl = Pipeline([("pre", pre_ctrl), ("clf", rf_ctrl)])
pipe_ctrl.fit(X_tr_control, y_tr)

# Impurity importance (built-in)
feature_names = pipe_ctrl.named_steps["pre"].get_feature_names_out()
impurity_imp = pd.Series(pipe_ctrl.named_steps["clf"].feature_importances_, index=feature_names).sort_values(ascending=False)

print("=== Top 10 by Impurity Importance ===")
print(impurity_imp.head(10))
print(f"\nrandom_id impurity rank: {list(impurity_imp.index).index([f for f in feature_names if 'random_id' in f][0]) + 1} out of {len(feature_names)}")

# Permutation importance (on validation set, more honest)
perm_result = permutation_importance(pipe_ctrl, X_val_control, y_val, n_repeats=10, random_state=42, n_jobs=1, scoring="average_precision")
perm_imp = pd.Series(perm_result.importances_mean, index=X_val_control.columns).sort_values(ascending=False)

print("\n=== Top 10 by Permutation Importance ===")
print(perm_imp.head(10))
print(f"\nrandom_id permutation rank: {list(perm_imp.index).index('random_id') + 1} out of {len(perm_imp)}")

=== Top 10 by Impurity Importance ===
num__TotalCharges                      0.129357
num__tenure                            0.122415
num__MonthlyCharges                    0.110556
num__random_id                         0.103971
cat__Contract_Month-to-month           0.053990
cat__OnlineSecurity_No                 0.030887
cat__PaymentMethod_Electronic check    0.027554
cat__TechSupport_No                    0.027282
cat__InternetService_Fiber optic       0.023244
num__SeniorCitizen                     0.016663
dtype: float64

random_id impurity rank: 4 out of 46

=== Top 10 by Permutation Importance ===
Contract            0.063439
tenure              0.045507
InternetService     0.036788
PaperlessBilling    0.016140
TotalCharges        0.006404
OnlineSecurity      0.004801
PaymentMethod       0.003835
TechSupport         0.003770
MultipleLines       0.001974
PhoneService        0.001391
dtype: float64

random_id permutation rank: 16 out of 20


## Impurity vs Permutation Importance: The Random ID Control

Planted a random, meaningless high-cardinality column (`random_id`) to test whether each importance method could be fooled by it.

**Impurity importance ranked `random_id` #4 out of 46** encoded features — ahead of `Contract`, `TechSupport`, and `OnlineSecurity`, despite carrying zero real information about churn. This confirms the known bias: impurity-based importance favors high-cardinality features simply because trees can split on them more often, regardless of whether those splits are meaningful.

**Permutation importance ranked `random_id` #16 out of 20** — correctly near the bottom, since shuffling it barely changes model performance.

**Conclusion:** Permutation importance should be preferred over impurity importance for interpretation, especially when high-cardinality categorical or ID-like columns are present. This is the exact mechanism behind the common real-world mistake of reporting a customer ID as a "top driver" of an outcome — a mistake this experiment reproduces on purpose to demonstrate why it happens.

In [59]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
impurity_imp.head(15).sort_values().plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("Top 15 by Impurity Importance")
perm_imp.head(15).sort_values().plot(kind="barh", ax=axes[1], color="coral")
axes[1].set_title("Top 15 by Permutation Importance")
fig.tight_layout()
fig.savefig("../reports/importance_comparison.png", dpi=120)
print("Saved to reports/importance_comparison.png")

Saved to reports/importance_comparison.png


In [63]:
import shap

In [64]:
# ---- SHAP beeswarm plot and single-customer explanation ----
# Use the tuned gradient boosting model from the Optuna study
X_tr_enc = pre_c.fit_transform(X_tr, y_tr)
X_val_enc = pre_c.transform(X_val)
feature_names_shap = pre_c.get_feature_names_out()

gb_final = HistGradientBoostingClassifier(random_state=42, **study.best_params)
gb_final.fit(X_tr_enc, y_tr)

explainer = shap.TreeExplainer(gb_final)
shap_values = explainer(X_val_enc[:500])  # sample for speed

shap.summary_plot(shap_values, X_val_enc[:500], feature_names=feature_names_shap, show=False, max_display=15)
plt.tight_layout()
plt.savefig("../reports/shap_beeswarm.png", dpi=120, bbox_inches="tight")
plt.close()
print("Saved to reports/shap_beeswarm.png")

# Top 3 features by mean absolute SHAP value
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
top3_idx = np.argsort(mean_abs_shap)[-3:][::-1]
print("\nTop 3 features by SHAP importance:")
for i in top3_idx:
    print(f"  {feature_names_shap[i]}: {mean_abs_shap[i]:.4f}")

C:\Users\samya\AppData\Local\Temp\ipykernel_14596\2878356893.py:13: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_values, X_val_enc[:500], feature_names=feature_names_shap, show=False, max_display=15)


Saved to reports/shap_beeswarm.png

Top 3 features by SHAP importance:
  cat__Contract_Month-to-month: 0.7354
  num__tenure: 0.2245
  num__MonthlyCharges: 0.2045


In [65]:
# ---- Single customer explanation ----
customer_idx = 0  # first customer in the validation sample

customer_shap = shap_values[customer_idx]
customer_pred_proba = gb_final.predict_proba(X_val_enc[customer_idx:customer_idx+1])[:, 1][0]

print(f"Customer's predicted churn probability: {customer_pred_proba:.3f}")
print(f"\nTop contributing factors (SHAP values):")

feature_shap_pairs = list(zip(feature_names_shap, customer_shap.values))
feature_shap_pairs_sorted = sorted(feature_shap_pairs, key=lambda x: abs(x[1]), reverse=True)

for feat, val in feature_shap_pairs_sorted[:5]:
    direction = "increased" if val > 0 else "decreased"
    print(f"  {feat}: {val:+.3f}  ({direction} churn risk)")

Customer's predicted churn probability: 0.532

Top contributing factors (SHAP values):
  cat__Contract_Month-to-month: +0.864  (increased churn risk)
  num__tenure: +0.488  (increased churn risk)
  cat__OnlineSecurity_No: +0.230  (increased churn risk)
  num__TotalCharges: +0.195  (increased churn risk)
  cat__TechSupport_No: +0.170  (increased churn risk)


In [66]:
# ---- Get this customer's actual raw feature values ----
customer_original = X_val.iloc[customer_idx]
print("Customer's actual values:")
print(f"  Contract: {customer_original['Contract']}")
print(f"  tenure: {customer_original['tenure']} months")
print(f"  OnlineSecurity: {customer_original['OnlineSecurity']}")
print(f"  TotalCharges: {customer_original['TotalCharges']}")
print(f"  TechSupport: {customer_original['TechSupport']}")

Customer's actual values:
  Contract: Month-to-month
  tenure: 2 months
  OnlineSecurity: No
  TotalCharges: 38.45
  TechSupport: No


## Single Customer Explanation

**Customer profile:** Month-to-month contract, 2 months tenure, no online security, no tech support, $38.45 total charges to date.

**Model's prediction:** 53.2% churn probability.

**Plain-English explanation:** This customer is flagged as moderately high-risk mainly because they are brand new (only 2 months with the company) and on a month-to-month contract — the easiest contract type to leave. Their lack of add-on services like online security and tech support means they have fewer reasons to stay attached to the company beyond the base service. None of these factors alone would push the risk this high, but together they compound: a new customer with no lock-in and no extra services is exactly the profile the model has learned to associate with leaving.

This illustrates how SHAP explanations work at the individual level — rather than a single "cause," multiple small, additive factors combine to produce the final prediction, and each one can be pointed to and explained without any of them individually being "the reason."

In [67]:
# ---- K-means clustering: choosing k with elbow and silhouette ----
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# Use only numeric features, scaled
numeric_cols_cluster = X.select_dtypes(include="number").columns.tolist()
X_numeric = X[numeric_cols_cluster].copy()
X_numeric = X_numeric.fillna(X_numeric.median())  # handle the 11 missing TotalCharges
X_numeric_scaled = StandardScaler().fit_transform(X_numeric)

inertias = []
silhouettes = []
k_range = range(2, 8)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_numeric_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_numeric_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(k_range), inertias, marker="o")
axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia")
axes[0].set_title("Elbow Method")

axes[1].plot(list(k_range), silhouettes, marker="o", color="orange")
axes[1].set_xlabel("k"); axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Method")

fig.tight_layout()
fig.savefig("../reports/kmeans_selection.png", dpi=120)
print("Saved to reports/kmeans_selection.png")

print("\nk | inertia | silhouette")
for k, inertia, sil in zip(k_range, inertias, silhouettes):
    print(f"{k} | {inertia:.1f} | {sil:.3f}")

Saved to reports/kmeans_selection.png

k | inertia | silhouette
2 | 16645.1 | 0.404
3 | 11241.7 | 0.461
4 | 8285.8 | 0.428
5 | 6477.2 | 0.447
6 | 4681.1 | 0.470
7 | 3843.4 | 0.450


In [68]:
# ---- Final clustering with k=3 ----
K_FINAL = 3
km_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
cluster_labels = km_final.fit_predict(X_numeric_scaled)

X_numeric["cluster"] = cluster_labels
X_numeric["Churn"] = y.values

print("Cluster profiles (mean values):")
print(X_numeric.groupby("cluster")[numeric_cols_cluster].mean().round(2))

print("\nCluster sizes and churn rates:")
print(X_numeric.groupby("cluster").agg(size=("Churn", "size"), churn_rate=("Churn", "mean")).round(3))

Cluster profiles (mean values):
         SeniorCitizen  tenure  MonthlyCharges  TotalCharges
cluster                                                     
0                  1.0   33.30           79.82       2810.47
1                  0.0   57.24           88.63       5056.32
2                  0.0   20.53           49.37        839.89

Cluster sizes and churn rates:
         size  churn_rate
cluster                  
0        1142       0.417
1        1875       0.145
2        4026       0.279


## K-Means Clustering (k=3)

Chosen k=3 based on silhouette score (0.461, among the higher values tested) and the point where inertia's rate of decrease begins to slow.

**Cluster 0 — "At-risk seniors"** (1,142 customers, 41.7% churn rate): Exclusively senior citizens, mid-tenure (~33 months), moderately high monthly charges (~$80). This group churns at nearly 3x the rate of the most loyal cluster despite not being the highest-paying segment — suggesting seniors may be more price-sensitive or under-served by current retention efforts.

**Cluster 1 — "Loyal high-value customers"** (1,875 customers, 14.5% churn rate): Long-tenured (~57 months), highest monthly charges (~$89) and highest total lifetime value (~$5,056). Despite paying the most, this group churns the least — classic evidence of successful lock-in, likely tied to long-term contracts.

**Cluster 2 — "New, price-sensitive customers"** (4,026 customers, 27.9% churn rate): Shortest tenure (~21 months) and lowest charges (~$49/mo). Churn rate sits close to the overall base rate (26.5%), consistent with this being the "typical," still-undifferentiated majority of the customer base.

**Honest assessment — did clustering add anything?** Yes, but modestly. The clusters reveal a genuine, actionable pattern (senior citizens churn substantially more than long-tenured high-value customers), which supervised feature importance methods (permutation importance, SHAP) did not surface on their own, since `SeniorCitizen` ranked low in both. However, the clusters are built purely from numeric features and don't use `Contract` or `TechSupport`, which we know from every other analysis in this project are the dominant churn drivers — so this clustering is a complementary lens, not a replacement for the supervised model's findings.

In [69]:
decisions_content = """# DECISIONS.md

## Framing
- **Decision sentence:** Because of this model, the retention team will call any customer whose predicted churn probability exceeds the chosen threshold, within 3 days of their next billing date.
- **Target:** Churn = 1 if Churn == "Yes", observed at the point of the snapshot in this dataset.
- **Base rate:** 26.54%

## Metric
- **Primary metric (classification):** PR-AUC, because the positive class (churn) is the minority class and accuracy would be misleading given the ~73.5% majority baseline.
- **Primary metric (regression, Stage 2):** MAE, to avoid letting a small number of high-tenure, high-bill customers dominate the score.

## Model & Threshold
- **Chosen model:** Logistic Regression (best validation PR-AUC: 0.663, most honest train/validation gap of any model tested)
- **Cost ratio:** Missing a churner (FN) costs 10x more than a wasted retention offer (FP), based on estimated customer lifetime value vs. contact cost.
- **Locked threshold:** 0.10

## Tuning
- **Trial budget (Optuna):** 60 trials, declared in advance
- **Search space:** learning_rate, max_leaf_nodes, min_samples_leaf, l2_regularization
- **Best tuned model (gradient boosting):** CV PR-AUC 0.6687, held-out PR-AUC 0.6723

## Locked Decision Rule
Contact customers whose predicted churn probability (from the logistic regression pipeline, trained on train+validation) exceeds 0.10.

Date locked: {date}
"""

from datetime import date
decisions_content = decisions_content.format(date=date.today().isoformat())

with open("../DECISIONS.md", "w") as f:
    f.write(decisions_content)

print("DECISIONS.md written and locked.")
print(decisions_content)

DECISIONS.md written and locked.
# DECISIONS.md

## Framing
- **Decision sentence:** Because of this model, the retention team will call any customer whose predicted churn probability exceeds the chosen threshold, within 3 days of their next billing date.
- **Target:** Churn = 1 if Churn == "Yes", observed at the point of the snapshot in this dataset.
- **Base rate:** 26.54%

## Metric
- **Primary metric (classification):** PR-AUC, because the positive class (churn) is the minority class and accuracy would be misleading given the ~73.5% majority baseline.
- **Primary metric (regression, Stage 2):** MAE, to avoid letting a small number of high-tenure, high-bill customers dominate the score.

## Model & Threshold
- **Chosen model:** Logistic Regression (best validation PR-AUC: 0.663, most honest train/validation gap of any model tested)
- **Cost ratio:** Missing a churner (FN) costs 10x more than a wasted retention offer (FP), based on estimated customer lifetime value vs. contact cost.


In [70]:
# ---- Stage 6: Refit on train+validation, evaluate on test ONCE ----
X_trval = pd.concat([X_tr, X_val])
y_trval = pd.concat([y_tr, y_val])

final_pipe = Pipeline([("pre", pre_c), ("clf", LogisticRegression(max_iter=1000, random_state=42))])
final_pipe.fit(X_trval, y_trval)

LOCKED_THRESHOLD = 0.10

# ---- Opening the test set, exactly once ----
proba_test = final_pipe.predict_proba(X_te)[:, 1]
pred_test = (proba_test >= LOCKED_THRESHOLD).astype(int)

test_pr_auc = average_precision_score(y_te, proba_test)
test_precision = precision_score(y_te, pred_test)
test_recall = recall_score(y_te, pred_test)
test_f1 = f1_score(y_te, pred_test)
test_cm = confusion_matrix(y_te, pred_test)

# Compare to validation performance at the same threshold, for the validation-test gap
pred_val_locked = (proba_val >= LOCKED_THRESHOLD).astype(int)
val_pr_auc_locked = average_precision_score(y_val, proba_val)

print("=== TEST SET RESULTS (opened once) ===")
print(f"PR-AUC: {test_pr_auc:.4f}")
print(f"Precision: {test_precision:.3f}")
print(f"Recall: {test_recall:.3f}")
print(f"F1: {test_f1:.3f}")
print(f"\nConfusion matrix:\n{test_cm}")
print(f"\nValidation PR-AUC (for comparison): {val_pr_auc_locked:.4f}")
print(f"Validation-test gap: {val_pr_auc_locked - test_pr_auc:.4f}")

=== TEST SET RESULTS (opened once) ===
PR-AUC: 0.6196
Precision: 0.400
Recall: 0.933
F1: 0.560

Confusion matrix:
[[512 523]
 [ 25 349]]

Validation PR-AUC (for comparison): 0.6779
Validation-test gap: 0.0584


## Stage 6: Test Set Results

The test set was opened exactly once, after refitting the final logistic regression pipeline on train+validation combined, using the locked threshold of 0.10 from `DECISIONS.md`.

**Results:**
- Test PR-AUC: 0.6196
- Precision: 0.400
- Recall: 0.933
- F1: 0.560
- Validation PR-AUC: 0.6779
- **Validation-test gap: 0.0584**

The gap between validation and test PR-AUC (0.0584) is real and reported without adjustment. This is larger than the near-zero gap observed in the earlier winner's-curse experiment on the tuned gradient boosting model, suggesting some optimism was present in the validation-stage results generally. Recall held up well on the test set (93.3%, close to validation), meaning the model's core strength — catching most churners — generalized. Precision (40.0%) means the retention team would need to accept a high false-alarm rate at this threshold, which is the direct trade-off of prioritizing recall via a low threshold under the assumed 10:1 cost ratio.

In [71]:
# ---- Business question: is the retention program worth it? ----
CONTACT_COST = 25
SUCCESS_RATE = 0.30
CUSTOMER_VALUE = 800

VALUE_PER_CAUGHT_CHURNER = SUCCESS_RATE * CUSTOMER_VALUE - CONTACT_COST  # $215
FP_COST = CONTACT_COST  # $25 wasted on someone who wouldn't have left

TP = test_cm[1, 1]  # correctly caught churners
FP = test_cm[0, 1]  # false alarms
FN = test_cm[1, 0]  # missed churners

targeted_value = TP * VALUE_PER_CAUGHT_CHURNER - FP * FP_COST
contacted_count = TP + FP

print("=== Targeted Strategy (model-driven, threshold=0.10) ===")
print(f"Customers contacted: {contacted_count}")
print(f"Churners caught: {TP}")
print(f"False alarms: {FP}")
print(f"Value: ${targeted_value:,.0f}")

# What if we just contacted everyone?
total_customers = len(y_te)
actual_churners = y_te.sum()
actual_stayers = total_customers - actual_churners

contact_everyone_value = actual_churners * VALUE_PER_CAUGHT_CHURNER - actual_stayers * FP_COST

print(f"\n=== Contact Everyone Strategy ===")
print(f"Customers contacted: {total_customers}")
print(f"Value: ${contact_everyone_value:,.0f}")

print(f"\n=== Comparison ===")
print(f"Targeted strategy value: ${targeted_value:,.0f}")
print(f"Contact-everyone value: ${contact_everyone_value:,.0f}")
winner = "Targeted strategy" if targeted_value > contact_everyone_value else "Contacting everyone"
print(f"Winner: {winner}")

=== Targeted Strategy (model-driven, threshold=0.10) ===
Customers contacted: 872
Churners caught: 349
False alarms: 523
Value: $61,960

=== Contact Everyone Strategy ===
Customers contacted: 1409
Value: $54,535

=== Comparison ===
Targeted strategy value: $61,960
Contact-everyone value: $54,535
Winner: Targeted strategy


## Business Question: Is the Retention Program Worth Running?

**Yes — and the model beats the naive alternative of contacting everyone.**

At the locked threshold (0.10), the targeted strategy contacts 872 customers, catches 349 of 374 actual churners, and generates an estimated **$61,960** in value (based on: catching a churner is worth $215 net after a 30% success rate and $25 contact cost; a false alarm costs $25 in wasted outreach).

**Compared to contacting everyone** (all 1,409 test customers): this would generate an estimated $54,535 — **$7,425 less** than the targeted approach, despite catching the same churners, because it wastes money contacting 1,035 customers who were never going to leave.

**Conclusion:** The retention program is worth running, and running it with the model's targeting is worth roughly $7,400 more per test-set-sized batch of customers than a blanket "contact everyone" policy — assuming the stated cost figures ($25 contact cost, $800 customer lifetime value, 30% offer success rate) hold in practice.

In [72]:
summary_content = """# Phase 2 Summary: Telco Customer Churn

## 1. Problem & Decision
Because of this model, the retention team will call any customer whose predicted churn probability exceeds 0.10, within 3 days of their next billing date. Base rate: 26.54% churn.

## 2. Data & Method
7,043 customers, 19 features after dropping the identifier column. Split 60/20/20, stratified, sealed until Stage 6. Same ColumnTransformer + Pipeline reused across every model (median/most-frequent imputation, standard scaling, one-hot encoding).

## 3. Baselines
| Baseline | Accuracy |
|---|---|
| Majority class | 0.7346 |
| Business rule (month-to-month + no tech support) | 0.7424 |
| Logistic regression (3 features) | 0.7906 |

## 4. Regression Detour (TotalCharges)
Random Forest hit a suspicious R²=0.999. Investigation confirmed `TotalCharges` ≈ tenure × MonthlyCharges; dropping MonthlyCharges only reduced R² to 0.995, since service columns still encode pricing information. Linear models (R²=0.908) were unaffected either way, confirming the leak is structural, not column-specific.

## 5. Classification & Threshold
Logistic Regression was the best and most honest classifier (validation PR-AUC 0.663, smallest train/validation gap). At the assumed 10:1 cost ratio (missed churner vs. wasted offer), the cost-optimal threshold is 0.10, cutting business cost 58% versus the default 0.5 threshold. Imbalance strategies (class_weight, SMOTE) matched each other but did not clearly beat simple threshold tuning on the plain model.

## 6. Ensembles & Tuning
Random Forest and Gradient Boosting did not beat Logistic Regression on validation PR-AUC, despite Random Forest reaching a perfect 1.000 on training data (severe overfitting, 0.376 gap). Optuna tuning (60 trials, declared in advance) improved Gradient Boosting to 0.6687 CV PR-AUC. The winner's-curse check showed a small negative gap (-0.0036), within normal fold-to-fold noise rather than clear evidence either way.

## 7. Interpretability
Permutation importance correctly ranked a planted random ID column near the bottom (#16/20), while impurity importance ranked it #4/46 — demonstrating impurity importance's known bias toward high-cardinality features. SHAP confirmed Contract, tenure, and MonthlyCharges as the top three drivers. K-means (k=3) revealed a genuine, actionable segment: senior citizens churn at 41.7% versus 14.5% for long-tenured high-value customers, a pattern not surfaced by the supervised feature importance methods.

## 8. Checkpoint Result & Business Answer
Test set opened once, after locking the decision rule. Test PR-AUC: 0.6196 (validation-test gap: 0.0584, reported without adjustment). The targeted strategy generates an estimated $61,960 in value on the test set, versus $54,535 for contacting every customer — a $7,425 improvement. **The retention program is worth running, and the model's targeting outperforms a blanket contact-everyone policy.**

## Limitations
The validation-test gap (0.0584) indicates some optimism in validation-stage model selection that did not fully carry to test. Cost assumptions ($25 contact cost, $800 customer value, 30% offer success rate) were not independently verified and materially affect the recommended threshold. Clustering used only numeric features and does not incorporate Contract or TechSupport, the two strongest churn signals identified elsewhere in this analysis.
"""

with open("../reports/phase2_summary.md", "w") as f:
    f.write(summary_content)

print("reports/phase2_summary.md written.")
print(summary_content)

reports/phase2_summary.md written.
# Phase 2 Summary: Telco Customer Churn

## 1. Problem & Decision
Because of this model, the retention team will call any customer whose predicted churn probability exceeds 0.10, within 3 days of their next billing date. Base rate: 26.54% churn.

## 2. Data & Method
7,043 customers, 19 features after dropping the identifier column. Split 60/20/20, stratified, sealed until Stage 6. Same ColumnTransformer + Pipeline reused across every model (median/most-frequent imputation, standard scaling, one-hot encoding).

## 3. Baselines
| Baseline | Accuracy |
|---|---|
| Majority class | 0.7346 |
| Business rule (month-to-month + no tech support) | 0.7424 |
| Logistic regression (3 features) | 0.7906 |

## 4. Regression Detour (TotalCharges)
Random Forest hit a suspicious R²=0.999. Investigation confirmed `TotalCharges` ≈ tenure × MonthlyCharges; dropping MonthlyCharges only reduced R² to 0.995, since service columns still encode pricing information. Linear mod

In [73]:
# ==== ORIGINAL BROKEN SCRIPT (for reference — do not run as-is) ====
"""
import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from imblearn.over_sampling import SMOTE

df = pd.read_csv("data/telco.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
df["customerID"] = df["customerID"].str.extract(r"(\\d+)").astype(float)  # "numeric feature"
y = (df["Churn"] == "Yes").astype(int)
X = pd.get_dummies(df.drop(columns=["Churn"]))

X = pd.DataFrame(StandardScaler().fit_transform(X), columns=X.columns)
X_bal, y_bal = SMOTE(random_state=42).fit_resample(X, y)
X_tr, X_te, y_tr, y_te = train_test_split(X_bal, y_bal, test_size=0.2, random_state=42)

grid = {"n_estimators":[100,300], "max_depth":[None,10,20], "min_samples_leaf":[1,3]}
gs = GridSearchCV(RandomForestClassifier(random_state=42), grid, cv=5, scoring="accuracy")
gs.fit(X_te, y_te)  # "tuning on the test set for a fair estimate"
model = gs.best_estimator_.fit(X_tr, y_tr)
pred = model.predict(X_te)
print("accuracy", accuracy_score(y_te, pred))
print("f1 ", f1_score(y_te, pred))

imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(imp.head(5))
"""
print("Reference script loaded as comment above.")

Reference script loaded as comment above.


In [75]:
# Run these lines from the reference script, in order, to reproduce the setup:
df = pd.read_csv("../data/Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
df["customerID"] = df["customerID"].str.extract(r"(\d+)").astype(float)
y_broken = (df["Churn"] == "Yes").astype(int)
X_broken = pd.get_dummies(df.drop(columns=["Churn"]))

print("Shape after get_dummies:", X_broken.shape)
print("Any text columns left?", X_broken.select_dtypes(include="object").columns.tolist())

Shape after get_dummies: (7043, 46)
Any text columns left? []


In [78]:
from sklearn.model_selection import GridSearchCV
# ---- Part B: Build the CORRECT pipeline first ----
df_audit = pd.read_csv("../data/Telco-Customer-Churn.csv")
df_audit["TotalCharges"] = pd.to_numeric(df_audit["TotalCharges"], errors="coerce")
df_audit = df_audit.drop(columns=["customerID"])  # dropped correctly, not numeric-ized

y_audit = (df_audit["Churn"] == "Yes").astype(int)
X_audit = df_audit.drop(columns=["Churn"])

# Correct split FIRST, before any preprocessing
X_tr_a, X_te_a, y_tr_a, y_te_a = train_test_split(
    X_audit, y_audit, test_size=0.2, stratify=y_audit, random_state=42
)

num_a = X_tr_a.select_dtypes(include="number").columns.tolist()
cat_a = X_tr_a.select_dtypes(exclude="number").columns.tolist()

pre_a = ColumnTransformer(transformers=[
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_a),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_a),
])

# Correct: SMOTE inside the pipeline, tuning only on training data, PR-AUC as the metric
from imblearn.pipeline import Pipeline as ImPipeline

correct_pipe = ImPipeline([
    ("pre", pre_a),
    ("smote", SMOTE(random_state=42)),
    ("clf", RandomForestClassifier(random_state=42)),
])

grid_a = {"clf__n_estimators": [100, 300], "clf__max_depth": [None, 10, 20], "clf__min_samples_leaf": [1, 3]}
gs_correct = GridSearchCV(correct_pipe, grid_a, cv=5, scoring="average_precision", n_jobs=1)
gs_correct.fit(X_tr_a, y_tr_a)  # tuned on TRAINING data only

proba_correct = gs_correct.predict_proba(X_te_a)[:, 1]
correct_pr_auc = average_precision_score(y_te_a, proba_correct)

print(f"CORRECT pipeline — Test PR-AUC: {correct_pr_auc:.4f}")
print(f"Base rate: {y_te_a.mean():.3f}")
print(f"Best params: {gs_correct.best_params_}")

CORRECT pipeline — Test PR-AUC: 0.6341
Base rate: 0.265
Best params: {'clf__max_depth': 10, 'clf__min_samples_leaf': 3, 'clf__n_estimators': 300}


In [79]:
# ---- Reintroduce Defect 1: scaling before split ----
X_scaled_before = X_audit.copy()
num_cols_all = X_scaled_before.select_dtypes(include="number").columns.tolist()

# Scale using ALL data (including what becomes test) before splitting
scaler_leak = StandardScaler()
X_scaled_before[num_cols_all] = scaler_leak.fit_transform(X_scaled_before[num_cols_all].fillna(X_scaled_before[num_cols_all].median()))

X_tr_d1, X_te_d1, y_tr_d1, y_te_d1 = train_test_split(
    X_scaled_before, y_audit, test_size=0.2, stratify=y_audit, random_state=42
)

# Rest of pipeline stays correct (SMOTE inside, tuned on train only)
pre_d1 = ColumnTransformer(transformers=[
    ("num", "passthrough", num_cols_all),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_a),
])

pipe_d1 = ImPipeline([
    ("pre", pre_d1),
    ("smote", SMOTE(random_state=42)),
    ("clf", RandomForestClassifier(random_state=42, **{k.replace("clf__", ""): v for k, v in gs_correct.best_params_.items()})),
])
pipe_d1.fit(X_tr_d1, y_tr_d1)
proba_d1 = pipe_d1.predict_proba(X_te_d1)[:, 1]
d1_pr_auc = average_precision_score(y_te_d1, proba_d1)

print(f"Defect 1 (scaling before split) — Test PR-AUC: {d1_pr_auc:.4f}")
print(f"Inflation vs correct: {d1_pr_auc - correct_pr_auc:+.4f}")

Defect 1 (scaling before split) — Test PR-AUC: 0.6329
Inflation vs correct: -0.0012


In [80]:
# ---- Reintroduce Defect 2: SMOTE before split ----
pre_d2 = ColumnTransformer(transformers=[
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_a),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_a),
])

X_audit_enc = pre_d2.fit_transform(X_audit, y_audit)
X_bal_d2, y_bal_d2 = SMOTE(random_state=42).fit_resample(X_audit_enc, y_audit)

X_tr_d2, X_te_d2, y_tr_d2, y_te_d2 = train_test_split(
    X_bal_d2, y_bal_d2, test_size=0.2, random_state=42
)

best_params_clean = {k.replace("clf__", ""): v for k, v in gs_correct.best_params_.items()}
model_d2 = RandomForestClassifier(random_state=42, **best_params_clean)
model_d2.fit(X_tr_d2, y_tr_d2)
proba_d2 = model_d2.predict_proba(X_te_d2)[:, 1]
d2_pr_auc = average_precision_score(y_te_d2, proba_d2)

print(f"Defect 2 (SMOTE before split) — Test PR-AUC: {d2_pr_auc:.4f}")
print(f"Inflation vs correct: {d2_pr_auc - correct_pr_auc:+.4f}")
print(f"NOTE: test set here contains synthetic rows and is class-balanced — not comparable to real base rate {y_audit.mean():.3f}")

Defect 2 (SMOTE before split) — Test PR-AUC: 0.9114
Inflation vs correct: +0.2773
NOTE: test set here contains synthetic rows and is class-balanced — not comparable to real base rate 0.265


In [81]:
# ---- Reintroduce Defect 3: tuning on the test set ----
X_tr_d3, X_te_d3, y_tr_d3, y_te_d3 = train_test_split(
    X_audit, y_audit, test_size=0.2, stratify=y_audit, random_state=42
)

pipe_d3 = ImPipeline([
    ("pre", pre_a),
    ("smote", SMOTE(random_state=42)),
    ("clf", RandomForestClassifier(random_state=42)),
])

gs_d3 = GridSearchCV(pipe_d3, grid_a, cv=5, scoring="average_precision", n_jobs=1)
gs_d3.fit(X_te_d3, y_te_d3)  # WRONG: tuning on test set

proba_d3 = gs_d3.predict_proba(X_te_d3)[:, 1]  # scoring on the same data it was tuned on
d3_pr_auc = average_precision_score(y_te_d3, proba_d3)

print(f"Defect 3 (tuned on test set) — Test PR-AUC: {d3_pr_auc:.4f}")
print(f"Inflation vs correct: {d3_pr_auc - correct_pr_auc:+.4f}")

Defect 3 (tuned on test set) — Test PR-AUC: 0.8560
Inflation vs correct: +0.2219


In [82]:
# ---- Reintroduce Defect 4: customerID as a numeric feature ----
df_d4 = pd.read_csv("../data/Telco-Customer-Churn.csv")
df_d4["TotalCharges"] = pd.to_numeric(df_d4["TotalCharges"], errors="coerce")
df_d4["customerID"] = df_d4["customerID"].str.extract(r"(\d+)").astype(float)  # kept as numeric feature

y_d4 = (df_d4["Churn"] == "Yes").astype(int)
X_d4 = df_d4.drop(columns=["Churn"])  # customerID NOT dropped

X_tr_d4, X_te_d4, y_tr_d4, y_te_d4 = train_test_split(
    X_d4, y_d4, test_size=0.2, stratify=y_d4, random_state=42
)

num_d4 = X_tr_d4.select_dtypes(include="number").columns.tolist()
cat_d4 = X_tr_d4.select_dtypes(exclude="number").columns.tolist()

pre_d4 = ColumnTransformer(transformers=[
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_d4),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_d4),
])

pipe_d4 = ImPipeline([
    ("pre", pre_d4),
    ("smote", SMOTE(random_state=42)),
    ("clf", RandomForestClassifier(random_state=42, **best_params_clean)),
])
pipe_d4.fit(X_tr_d4, y_tr_d4)
proba_d4 = pipe_d4.predict_proba(X_te_d4)[:, 1]
d4_pr_auc = average_precision_score(y_te_d4, proba_d4)

print(f"Defect 4 (customerID as feature) — Test PR-AUC: {d4_pr_auc:.4f}")
print(f"Inflation vs correct: {d4_pr_auc - correct_pr_auc:+.4f}")

# Show where customerID ranks in feature importance — this is the REAL problem, not the score
feature_names_d4 = pre_d4.get_feature_names_out()
importances_d4 = pipe_d4.named_steps["clf"].feature_importances_
imp_series_d4 = pd.Series(importances_d4, index=feature_names_d4).sort_values(ascending=False)
customerID_rank = list(imp_series_d4.index).index([f for f in feature_names_d4 if "customerID" in f][0]) + 1
print(f"\ncustomerID importance rank: {customerID_rank} out of {len(feature_names_d4)}")
print(imp_series_d4.head(5))

Defect 4 (customerID as feature) — Test PR-AUC: 0.6234
Inflation vs correct: -0.0107

customerID importance rank: 10 out of 46
cat__Contract_Month-to-month           0.128353
cat__OnlineSecurity_No                 0.085466
num__tenure                            0.084726
cat__PaymentMethod_Electronic check    0.069862
cat__TechSupport_No                    0.062318
dtype: float64


In [84]:
from sklearn.metrics import accuracy_score
# ---- Reintroduce Defect 5: accuracy instead of PR-AUC ----
# Using the CORRECT pipeline's predictions, just changing what metric is reported
pred_correct_05 = (proba_correct >= 0.5).astype(int)
acc_correct = accuracy_score(y_te_a, pred_correct_05)
f1_correct = f1_score(y_te_a, pred_correct_05)

print(f"If we reported accuracy instead of PR-AUC:")
print(f"Accuracy: {acc_correct:.4f}  (sounds great!)")
print(f"F1: {f1_correct:.4f}")
print(f"But base rate is {y_te_a.mean():.3f} — majority-class-only accuracy would be {1-y_te_a.mean():.3f}")
print(f"True PR-AUC: {correct_pr_auc:.4f}")

If we reported accuracy instead of PR-AUC:
Accuracy: 0.7665  (sounds great!)
F1: 0.6097
But base rate is 0.265 — majority-class-only accuracy would be 0.735
True PR-AUC: 0.6341


In [85]:
# ---- Part C: The corrected result with a cost-chosen threshold ----
def total_cost_audit(y_true, proba, threshold, fn_cost, fp_cost):
    pred = (proba >= threshold).astype(int)
    fn = ((y_true == 1) & (pred == 0)).sum()
    fp = ((y_true == 0) & (pred == 1)).sum()
    return fn * fn_cost + fp * fp_cost

FN_COST = 10  # same assumption as Assignment 1: missing a churner costs 10x a false alarm
FP_COST = 1

thresholds_audit = np.linspace(0.05, 0.95, 91)
costs_audit = [total_cost_audit(y_te_a, proba_correct, t, FN_COST, FP_COST) for t in thresholds_audit]
best_threshold_audit = thresholds_audit[np.argmin(costs_audit)]

pred_final = (proba_correct >= best_threshold_audit).astype(int)
final_precision = precision_score(y_te_a, pred_final)
final_recall = recall_score(y_te_a, pred_final)

print(f"Cost-optimal threshold: {best_threshold_audit:.2f}")
print(f"Precision: {final_precision:.3f}")
print(f"Recall: {final_recall:.3f}")
print(f"PR-AUC: {correct_pr_auc:.4f}")
print(f"Base rate: {y_te_a.mean():.3f}")

Cost-optimal threshold: 0.18
Precision: 0.401
Recall: 0.949
PR-AUC: 0.6341
Base rate: 0.265


In [86]:
audit_content = f"""# Model Audit: Churn Prediction Script Review

## Headline

The reported figure was **96.4% accuracy, F1 of 0.94**. The honest figure, on a correctly built pipeline, is **PR-AUC of 0.6341** (precision 0.401, recall 0.949 at a cost-chosen threshold of 0.18). The reported numbers were inflated primarily by two compounding leaks — SMOTE applied before the train/test split, and hyperparameter tuning performed directly on the test set — each independently inflating PR-AUC by roughly 0.22–0.28 when isolated and measured.

## Defect Table (ranked by impact)

| # | Defect | Correct | With defect | Inflation | Verdict |
|---|---|---|---|---|---|
| 1 | SMOTE before split | 0.6341 | 0.9114 | +0.2773 | Fatal |
| 2 | Tuned on test set | 0.6341 | 0.8560 | +0.2219 | Fatal |
| 3 | Accuracy on imbalanced data | 0.7665 (acc) | — | — | Misleading (73.5% is the do-nothing floor) |
| 4 | customerID as a numeric feature | 0.6341 | 0.6234 | -0.0107 | Distorts meaning, not score |
| 5 | Scaling fitted before split | 0.6341 | 0.6329 | -0.0012 | Small but real (model-dependent) |
| 6 | Impurity importance interpreted causally | N/A | N/A | N/A | Distorts interpretation |
| 7 | GridSearchCV scored on accuracy | N/A | N/A | N/A | Distorts interpretation |
| 8 | "Target high-ID customers" recommendation | N/A | N/A | N/A | Distorts interpretation (conclusion-only defect, not in the code) |

## The Most Serious Defect, Explained Plainly

The script balanced the dataset with a technique called SMOTE before splitting it into training and test sets, rather than after. This means some of the test examples were built using information from what should have been unseen data, similar to letting a student see practice questions that are near-copies of their final exam. As a result, the reported 96.4% accuracy reflects the model's ability to recognize data it effectively already saw, not its ability to predict churn for genuinely new customers.

## What the Model Is Actually Worth

At the cost-chosen threshold of 0.18 (assuming a missed churner costs roughly 10x more than a wasted retention contact), the corrected model catches 94.9% of actual churners at 40.1% precision. This is a real, usable result — it beats doing nothing — but it is far more modest than the original 96.4%/94% figures suggested, and it should be evaluated against that honest baseline, not the inflated one.

## Process Recommendation

Before any model result reaches a management recommendation, require that the train/test split happen first, before any resampling, scaling, or tuning step touches the data — and that this be checked by a second person reviewing the code, not just the printed metrics. A simple pre-deployment checklist item ("was the test set touched more than once, and was it touched before the final evaluation?") would have caught both fatal defects in this script before they reached a business decision.
"""

with open("../reports/audit.md", "w") as f:
    f.write(audit_content)

print("reports/audit.md written.")
print(audit_content)

reports/audit.md written.
# Model Audit: Churn Prediction Script Review

## Headline

The reported figure was **96.4% accuracy, F1 of 0.94**. The honest figure, on a correctly built pipeline, is **PR-AUC of 0.6341** (precision 0.401, recall 0.949 at a cost-chosen threshold of 0.18). The reported numbers were inflated primarily by two compounding leaks — SMOTE applied before the train/test split, and hyperparameter tuning performed directly on the test set — each independently inflating PR-AUC by roughly 0.22–0.28 when isolated and measured.

## Defect Table (ranked by impact)

| # | Defect | Correct | With defect | Inflation | Verdict |
|---|---|---|---|---|---|
| 1 | SMOTE before split | 0.6341 | 0.9114 | +0.2773 | Fatal |
| 2 | Tuned on test set | 0.6341 | 0.8560 | +0.2219 | Fatal |
| 3 | Accuracy on imbalanced data | 0.7665 (acc) | — | — | Misleading (73.5% is the do-nothing floor) |
| 4 | customerID as a numeric feature | 0.6341 | 0.6234 | -0.0107 | Distorts meaning, not score |
| 